In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [4]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((455, 30), (114, 30), (455,), (114,))

In [5]:
feature_summary = (
    X_train.describe()
    .T[['mean', 'std', 'min', 'max']]
    .rename(columns={"mean": "Mean", "std": "Std", "min": "Min", "max": "Max"})
    .round(2)
)
feature_summary


,Mean,Std,Min,Max
mean radius,14.07,3.50,6.98,28.11
mean texture,19.25,4.41,9.71,39.28
mean perimeter,91.56,24.15,43.79,188.50
mean area,648.54,344.94,143.50,2499.00
mean smoothness,0.10,0.01,0.06,0.14
mean compactness,0.10,0.05,0.02,0.35
mean concavity,0.09,0.08,0.00,0.43
mean concave points,0.05,0.04,0.00,0.20
mean symmetry,0.18,0.03,0.11,0.30
mean fractal dimension,0.06,0.01,0.05,0.10


In [6]:
correlation = X_train.corrwith(y_train).sort_values(ascending=False)
correlation


smoothness error           0.070507
texture error              0.023917
mean fractal dimension    -0.001027
symmetry error            -0.012765
fractal dimension error   -0.059216
concavity error           -0.252785
compactness error         -0.291239
worst fractal dimension   -0.327308
mean symmetry             -0.358370
mean smoothness           -0.392200
mean texture              -0.409384
concave points error      -0.418560
worst smoothness          -0.429227
worst symmetry            -0.430422
worst texture             -0.450765
perimeter error           -0.551905
radius error              -0.565074
area error                -0.567197
worst compactness         -0.602331
mean compactness          -0.610722
worst concavity           -0.668368
mean concavity            -0.692052
mean area                 -0.713409
mean radius               -0.732224
worst area                -0.737965
mean perimeter            -0.745132
worst radius              -0.776390
mean concave points       -0

In [7]:
class_balance = pd.DataFrame({
    "count": y_train.value_counts().sort_index(),
    "share": y_train.value_counts(normalize=True).sort_index() * 100
}).rename(index={0: "malignant", 1: "benign"})
class_balance["share"] = class_balance["share"].round(2)
class_balance


,count,share
target,,
malignant,170,37.36
benign,285,62.64


In [ ]:
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba_malignant = pipe.predict_proba(X_test)[:, 0]  # вероятность класса 'malignant'
